# Demo 01 : What agents share

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=500, max_retries=0)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
NOTEBOOK_DIR = ROOT / "notebooks" / "06-agent-communication" if ROOT else Path.cwd()


def stock(part="CE-CF-02"):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?", (part,)).fetchone()
    return dict(row) if row else {"part_no": part, "error": "unknown part"}


def guidance(error="F-CE-110"):
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error + " |"))


FACTS = stock()
QUESTION = "Report CE-CF-02 stock and location, then the first action and replacement condition for F-CE-110."
LIMIT = {"recursion_limit": 16}


def finish_metrics(name):
    row = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "2")),
           "calls": counter.calls, "input_tokens": counter.input_tokens,
           "output_tokens": counter.output_tokens, "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", row)
    (NOTEBOOK_DIR / (name + "-metrics.json")).write_text(json.dumps(row, indent=2) + "\n", encoding="utf-8")

## a · shared-state-or-messages

We give a parts worker and a manuals worker the same maintenance request. A shared-state graph stores their findings in separate keys.

In [3]:
class Shared(TypedDict):
    question: str
    stock: dict
    guidance: str
    answer: str


def combine(state):
    return {"answer": llm.invoke(state["question"] + "\nEvidence only:\n"
                                + json.dumps(state["stock"]) + "\n" + state["guidance"]).content}


builder = StateGraph(Shared)
builder.add_node("parts", lambda s: {"stock": stock()})
builder.add_node("manuals", lambda s: {"guidance": guidance()})
builder.add_node("combine", combine)
builder.add_edge(START, "parts")
builder.add_edge(START, "manuals")
builder.add_edge(["parts", "manuals"], "combine")
builder.add_edge("combine", END)
shared = builder.compile().invoke({"question": QUESTION})
print("shared keys :", sorted(shared))
print("answer :", shared["answer"])

shared keys : ['answer', 'guidance', 'question', 'stock']
answer : CE-CF-02 has **6 in stock** at **Rietpoort**. For F-CE-110, first **check the traction cooling fan**; replace it if airflow is **below 80%**.


We then pass an explicit message envelope. The receiver gets the payload we put in it. Shared state would also expose the caller's other fields. This local Python example has no network transport.

In [4]:
class Envelope(TypedDict):
    sender: str
    recipient: str
    correlation_id: str
    payload: dict


def receive(envelope: Envelope):
    print("received :", envelope)
    return llm.invoke("Write a stock note from this payload only.\n" + json.dumps(envelope["payload"])).content


envelope = Envelope(sender="parts", recipient="coordinator", correlation_id="BW-7001", payload=FACTS)
print("message answer :", receive(envelope))

received : {'sender': 'parts', 'recipient': 'coordinator', 'correlation_id': 'BW-7001', 'payload': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}


message answer : Stock note: Rietpoort has 6 traction cooling fans (part no. CE-CF-02) in stock, priced at €420.00 each.


## b · full-history-task-or-summary

We compare full history, task-only input, a copied history with a new task, and a selected factual summary. The missing first-action request is deliberate in the task-only case.

In [5]:
history = [
    HumanMessage("Bus CE12-2021-0117 has F-CE-110. Include the first action and replacement condition."),
    AIMessage("Stock lookup complete. " + json.dumps(FACTS)),
    HumanMessage("Write the final maintenance note."),
]
task_only = [HumanMessage("Write a CE-CF-02 stock note. " + json.dumps(FACTS))]
fork = [*history, HumanMessage("Assess the inspection-first option. Preserve conditions.")]
summary = [HumanMessage("Maintenance case BW-7001. " + QUESTION + "\n" + json.dumps(FACTS))]
contexts = {"full history": history, "task only": task_only, "fork": fork, "summary": summary}
prompt = SystemMessage("Use visible facts only. Do not infer repair completion. Manual excerpt:\n" + guidance())
for name, messages in contexts.items():
    reply = llm.invoke([prompt, *messages])
    print("context :", name, [(m.type, m.content) for m in messages])
    print("answer :", reply.content)
    print("input tokens :", (reply.usage_metadata or {}).get("input_tokens"))
print("parent history :", len(history), "fork history :", len(fork))

context : full history [('human', 'Bus CE12-2021-0117 has F-CE-110. Include the first action and replacement condition.'), ('ai', 'Stock lookup complete. {"part_no": "CE-CF-02", "name": "traction cooling fan", "stock": 6, "price_eur": 420.0, "depot": "Rietpoort"}'), ('human', 'Write the final maintenance note.')]
answer : Bus CE12-2021-0117 has fault code F-CE-110 (traction system over-temperature). First, check the traction cooling fan. Replace it if airflow is below 80 percent.
input tokens : 149


context : task only [('human', 'Write a CE-CF-02 stock note. {"part_no": "CE-CF-02", "name": "traction cooling fan", "stock": 6, "price_eur": 420.0, "depot": "Rietpoort"}')]
answer : CE-CF-02 — Traction cooling fan  
Stock: 6 units | Price: €420.00 each | Depot: Rietpoort
input tokens : 115


context : fork [('human', 'Bus CE12-2021-0117 has F-CE-110. Include the first action and replacement condition.'), ('ai', 'Stock lookup complete. {"part_no": "CE-CF-02", "name": "traction cooling fan", "stock": 6, "price_eur": 420.0, "depot": "Rietpoort"}'), ('human', 'Write the final maintenance note.'), ('human', 'Assess the inspection-first option. Preserve conditions.')]
answer : The manual’s first action for F-CE-110 is to check the traction cooling fan. Replace it only if airflow is below 80 percent. The stock lookup lists 6 traction cooling fans (CE-CF-02) at the Rietpoort depot, priced at €420 each. No inspection result or repair completion was provided.
input tokens : 162


context : summary [('human', 'Maintenance case BW-7001. Report CE-CF-02 stock and location, then the first action and replacement condition for F-CE-110.\n{"part_no": "CE-CF-02", "name": "traction cooling fan", "stock": 6, "price_eur": 420.0, "depot": "Rietpoort"}')]
answer : CE-CF-02 (traction cooling fan): stock 6, at the Rietpoort depot. For F-CE-110, first check the traction cooling fan; replace it if airflow is below 80%.
input tokens : 136
parent history : 3 fork history : 4


We copy the history in ordinary Python to expose the fork contract. No checkpoint fork or Deep Agents harness runs here. A summary can lose a condition at either boundary, so we inspect the worker's return before using it.

In [6]:
worker_answer = llm.invoke([prompt, *summary]).content
returned = {"case": "BW-7001", "worker": "maintenance", "summary": worker_answer}
print("summary back :", returned)
print("caller sees :", [history[-1].content, returned])

summary back : {'case': 'BW-7001', 'worker': 'maintenance', 'summary': 'CE-CF-02 traction cooling fan: stock 6 at the Rietpoort depot.\n\nFor F-CE-110, first check the traction cooling fan. Replace it if airflow is below 80%.'}
caller sees : ['Write the final maintenance note.', {'case': 'BW-7001', 'worker': 'maintenance', 'summary': 'CE-CF-02 traction cooling fan: stock 6 at the Rietpoort depot.\n\nFor F-CE-110, first check the traction cooling fan. Replace it if airflow is below 80%.'}]


## Things to try

Optional : we can change the part, the context boundary or the policy and inspect the result.

In [7]:
finish_metrics("demo01")

metrics : {'model': 'openai/gpt-6-luna', 'rps': 0.4, 'calls': 7, 'input_tokens': 871, 'output_tokens': 332, 'seconds': 19.4}
